<style>
.cl-responsive,
.cl-responsive * {
  box-sizing: border-box !important;
  max-width: 100% !important;
}
.cl-responsive {
  width: 100% !important;
  overflow-x: hidden !important;
  white-space: normal !important;
  overflow-wrap: anywhere !important;
  word-break: normal !important;
}
.cl-responsive code {
  white-space: normal !important;
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}
.cl-cover h1 {
  font-size: clamp(28px, 4vw, 42px) !important;
  line-height: 1.15 !important;
}
.cl-cover p {
  max-width: min(820px, 100%) !important;
}
</style>

<div class="cl-responsive cl-cover" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;padding:32px 36px;border-radius:16px;background:linear-gradient(135deg,#102A43 0%,#243B53 100%);color:#F7F4EC;margin:0 0 24px 0;">
  <div style="font-size:13px;letter-spacing:.16em;text-transform:uppercase;color:#D4A44C;font-weight:700;overflow-wrap:anywhere;">Exploratory Data Analysis</div>
  <h1 style="margin:10px 0 8px 0;font-size:clamp(28px,4vw,42px);line-height:1.15;font-weight:700;color:#FFFFFF;overflow-wrap:anywhere;word-break:normal;">ConflictLens — Foundation EDA</h1>
  <p style="margin:0;max-width:min(820px,100%);font-size:18px;line-height:1.55;color:#D9E2EC;overflow-wrap:anywhere;word-break:normal;">Inventory, profile and compare the available conflict-related datasets before making any analytical design decision.</p>
  <div style="margin-top:24px;padding-top:16px;border-top:1px solid rgba(255,255,255,.20);font-size:13px;color:#BCCCDC;overflow-wrap:anywhere;word-break:normal;">Python · Polars · Plotly · Data Quality · Conflict Data</div>
</div>


## Scope

This notebook is strictly limited to **exploratory data analysis and technical profiling**. It does not select a final dataset, recommend a V1 source, or make causal claims about conflict dynamics.

The working principle is:

```text
Observe → profile → compare → document open questions
```

The objective is to build a reliable foundation for future analytical work by clarifying what each source can and cannot support.

<a id="executive-summary"></a>

## Executive summary

This notebook provides the first technical foundation for the ConflictLens project. It inventories, profiles and compares the available conflict-related datasets before selecting any analytical source or defining a V1 dataset.

The current data repository contains **63 files** for approximately **504.91 MB**. Storage volume is mainly driven by CSV files, especially the V-Dem country-year file, which is the largest single file in the repository at approximately **387.53 MB**.

The notebook loads or profiles **18 analytical tables**, representing **1,709,079 observed rows**. The largest table by row count is **ACLED regional aggregates**, with **965,913 rows**. The widest full schema is **V-Dem**, with **4,618 columns**, which requires selective loading for stable exploratory work.

The datasets are not directly comparable by default. They differ in temporal coverage, geographic conventions, units of observation, identifiers and data collection logic. Some sources are event-level tables, while others are country-year panels, actor-year records, admin-week aggregates or origin-asylum-year migration tables.

The main EDA conclusion is methodological: ConflictLens has a strong data foundation, but future analysis must be sequenced around clear units of observation. The recommended next step is to build a country-year analytical panel first, while preserving event-level analysis as a second, complementary track.

## Navigation

1. [Executive summary](#executive-summary)
2. [Project framing](#project-framing)
3. [Environment and loading strategy](#environment)
4. [Global file inventory](#file-inventory)
5. [Dataset loading](#dataset-loading)
6. [Technical profiling](#technical-profiling)
7. [Temporal coverage](#temporal-coverage)
8. [Geographic coverage](#geographic-coverage)
9. [Granularity and identifiers](#granularity-identifiers)
10. [Data quality checks](#data-quality)
11. [First descriptive exploration](#first-descriptive-exploration)
12. [Join feasibility diagnostics](#join-feasibility)
13. [Comparative EDA matrix](#comparative-eda-matrix)
14. [Conclusion and analytical strategy](#conclusion-analytical-strategy)

<a id="project-framing"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 1</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Project framing</h2>
</div>

The objective is to understand the **structure**, **coverage**, **granularity**, **identifier availability** and **quality risks** of the datasets currently available in the repository.

This notebook deliberately stops before source selection. It documents what can be observed from the files, what is technically comparable, and which methodological constraints must be resolved before substantive interpretation.

<a id="environment"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 2</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Environment and loading strategy</h2>
</div>

Expected repository layout:

```text
repo_root/
├── 01_conflictlens_foundation_eda.ipynb
└── data/
    ├── acled/
    ├── EPR/
    ├── svac/
    ├── ucdp/
    ├── UNHCR_RefugeeDataFinder_Copyright/
    ├── UNHCR_RefugeeDataFinder_Copyright - demo/
    ├── V-Dem-CY-FullOthers-v16_csv/
    └── wdi_downloader_package/
```

Required libraries:

```powershell
python -m pip install polars pyarrow fastexcel plotly nbformat ipykernel
```

The notebook uses relative paths only. Run it from the repository root.

In [1]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import importlib.util
import platform
import sys
from pathlib import Path
from zipfile import ZipFile
from typing import Any
import re
import csv
import warnings

from IPython.display import HTML, Markdown, display

REQUIRED_MODULES = {
    "polars": "polars",
    "plotly": "plotly",
    "fastexcel": "fastexcel",
    "pyarrow": "pyarrow",
}

missing = [module for module in REQUIRED_MODULES if importlib.util.find_spec(module) is None]
if missing:
    packages = " ".join(REQUIRED_MODULES[module] for module in missing)
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n\n"
        "Install the project requirements from the repository root:\n"
        "    python -m pip install -r requirements_conflictlens_eda.txt\n\n"
        "Or install the missing packages directly:\n"
        f"    python -m pip install {packages}"
    )

import polars as pl
import plotly.express as px
import plotly.io as pio

warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype"
RENDER_INLINE_PLOTS = True  # Set to False to save Plotly charts as HTML links instead of rendering them inline.

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_thousands_separator(",")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    for candidate in [*PROJECT_ROOT.parents]:
        if (candidate / "data").exists():
            PROJECT_ROOT = candidate
            break

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PLOT_DIR = OUTPUT_DIR / "plotly_html"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "data/ not found. Run the notebook from the repository root, or place it in a folder under the repository root. "
        "Expected structure: repo_root/data/."
    )

package_versions = {module: importlib_metadata.version(package) for module, package in REQUIRED_MODULES.items()}

display(Markdown(
    "<div style='padding:14px 16px;border-left:4px solid #4C78A8;background:#DCEEFF;border-radius:6px;line-height:1.55;'>"
    "<strong>Execution environment check.</strong><br>"
    f"Python: <code>{sys.version.split()[0]}</code> · Platform: <code>{platform.platform()}</code><br>"
    f"Project root resolved to: <code>{PROJECT_ROOT}</code><br>"
    f"Data directory exists: <code>{DATA_DIR.exists()}</code> · Path: <code>{DATA_DIR}</code><br>"
    "Packages: " + ", ".join([f"<code>{k} {v}</code>" for k, v in package_versions.items()]) +
    "</div>"
))

<div style='padding:14px 16px;border-left:4px solid #4C78A8;background:#DCEEFF;border-radius:6px;line-height:1.55;'><strong>Execution environment check.</strong><br>Python: <code>3.13.5</code> · Platform: <code>Linux-4.4.0-x86_64-with-glibc2.41</code><br>Project root resolved to: <code>./extracted</code><br>Data directory exists: <code>True</code> · Path: <code>./extracted/data</code><br>Packages: <code>polars 1.42.1</code>, <code>plotly 6.5.2</code>, <code>fastexcel 0.20.2</code>, <code>pyarrow 24.0.0</code></div>

In [2]:
def size_mb(path: Path) -> float:
    return round(path.stat().st_size / 1024**2, 2)


def normalize_name(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("’", "'")
    return text or None


def to_pandas(df: pl.DataFrame):
    """Adapter for Plotly. Data processing remains in Polars."""
    return df.to_pandas()


NOTE_CSS = """<style>
.cl-note,
.cl-note * {
    box-sizing: border-box !important;
    max-width: 100% !important;
}
.cl-note {
    box-sizing: border-box !important;
    width: 100% !important;
    max-width: 100% !important;
    overflow-x: hidden !important;
    white-space: normal !important;
    overflow-wrap: anywhere !important;
    word-break: normal !important;
}
.cl-note code {
    white-space: normal !important;
    overflow-wrap: anywhere !important;
    word-break: break-word !important;
}
</style>"""

NOTE_STYLE = (
    "box-sizing:border-box;"
    "width:100%;"
    "max-width:100%;"
    "overflow-x:hidden;"
    "padding:14px 16px;"
    "border-left:4px solid #4C78A8;"
    "background:#DCEEFF;"
    "border-radius:6px;"
    "line-height:1.55;"
    "white-space:normal;"
    "overflow-wrap:anywhere;"
    "word-break:normal;"
)


def display_note(text: str) -> None:
    display(HTML(f"{NOTE_CSS}<div class='cl-note' style='{NOTE_STYLE}'>{text}</div>"))


def show_plot(fig, filename: str) -> None:
    """Render a Plotly chart inline or save it as an interactive HTML file.

    Inline rendering is disabled by default to keep the executed notebook lightweight.
    Set RENDER_INLINE_PLOTS = True near the top if you want charts to appear inline.
    """
    if RENDER_INLINE_PLOTS:
        fig.show()
    else:
        html_path = PLOT_DIR / f"{filename}.html"
        fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
        rel = html_path.relative_to(PROJECT_ROOT).as_posix()
        display(Markdown(f"Interactive Plotly chart saved to: [`{rel}`]({rel})"))


def read_csv_file(path: Path, *, infer_schema_length: int = 10_000) -> pl.DataFrame:
    return pl.read_csv(path, infer_schema_length=infer_schema_length, ignore_errors=True)


def zip_csv_names(path: Path) -> list[str]:
    with ZipFile(path) as archive:
        return [name for name in archive.namelist() if name.lower().endswith(".csv")]


def zip_csv_header(path: Path, inner_name: str | None = None, encoding: str = "utf-8") -> list[str]:
    with ZipFile(path) as archive:
        selected = inner_name or zip_csv_names(path)[0]
        with archive.open(selected) as file:
            return file.readline().decode(encoding, errors="replace").strip().split(",")


def read_csv_from_zip(path: Path, inner_name: str | None = None, columns: list[str] | None = None) -> pl.DataFrame:
    with ZipFile(path) as archive:
        selected = inner_name or zip_csv_names(path)[0]
        with archive.open(selected) as file:
            try:
                return pl.read_csv(file, columns=columns, infer_schema_length=10_000, ignore_errors=True)
            except Exception:
                file.seek(0)
                return pl.read_csv(file, columns=columns, infer_schema_length=10_000, ignore_errors=True, encoding="utf8-lossy")


def read_excel_file(path: Path) -> pl.DataFrame:
    return pl.read_excel(path)


def safe_min_max(df: pl.DataFrame, col: str) -> tuple[Any, Any]:
    out = df.select(pl.col(col).min().alias("min_value"), pl.col(col).max().alias("max_value")).to_dicts()[0]
    return out["min_value"], out["max_value"]


def percent(value: int | float | None, total: int | float | None) -> float | None:
    if value is None or total in (None, 0):
        return None
    return round(float(value) / float(total) * 100, 2)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — environment and loading strategy</strong>
<br><br>
<strong>Analytical question.</strong> Can the notebook be executed locally without depending on machine-specific paths or hidden configuration?
<br>
<strong>Method.</strong> The first execution cell checks required dependencies, resolves the project root from the current working directory, and verifies that the relative <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">data/</code> folder is available. Utility functions centralize ZIP/CSV reading, Excel loading, Plotly export and minimal country-name normalization.
<br>
<strong>Observed result.</strong> During execution, the required dependencies are available and the <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">data/</code> directory is found before any analytical source is loaded.
<br>
<strong>Interpretation.</strong> This makes the notebook more reproducible. If execution fails locally, the most likely causes are explicit: a missing dependency or a notebook launched outside the repository structure.
<br>
<strong>EDA implication.</strong> Data loading should only start after the execution environment has been validated.
</div>

<a id="file-inventory"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 3</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Global file inventory</h2>
</div>

In [3]:
all_files = [path for path in DATA_DIR.rglob("*") if path.is_file()]
file_inventory = pl.DataFrame([
    {
        "relative_path": str(path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "directory": str(path.parent.relative_to(DATA_DIR)).replace("\\", "/"),
        "filename": path.name,
        "extension": path.suffix.lower().lstrip(".") or "no_extension",
        "size_mb": size_mb(path),
    }
    for path in all_files
]).sort(["directory", "filename"])

file_summary = (
    file_inventory
    .group_by("extension")
    .agg(pl.len().alias("n_files"), pl.sum("size_mb").round(2).alias("total_size_mb"))
    .sort("total_size_mb", descending=True)
)

top_files = file_inventory.sort("size_mb", descending=True).head(12)

print(f"Files found under data/: {file_inventory.height}")
print(f"Total size under data/: {file_inventory['size_mb'].sum():,.2f} MB")
file_summary

Files found under data/: 63
Total size under data/: 504.91 MB


extension,n_files,total_size_mb
str,u32,f64
"""csv""",15,412.42
"""xlsx""",13,42.56
"""zip""",4,37.57
"""pdf""",12,11.06
"""parquet""",1,0.38
"""js""",8,0.3
"""css""",2,0.3
"""htm""",1,0.17
"""md""",3,0.1


In [4]:
fig = px.bar(
    to_pandas(file_summary), x="extension", y="total_size_mb", text="n_files",
    title="File volume by extension",
    labels={"extension":"Extension", "total_size_mb":"Total size (MB)", "n_files":"Number of files"},
)
fig.update_layout(height=420)
show_plot(fig, "chart_02")

In [5]:
top_files

relative_path,directory,filename,extension,size_mb
str,str,str,str,f64
"""data/V-Dem-CY-FullOthers-v16_csv/V-Dem-CY-Full+Others-v16.csv""","""V-Dem-CY-FullOthers-v16_csv""","""V-Dem-CY-Full+Others-v16.csv""","""csv""",387.53
"""data/ucdp/ged261-csv.zip""","""ucdp""","""ged261-csv.zip""","""zip""",37.31
"""data/acled/Africa_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""acled""","""Africa_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""xlsx""",11.6
"""data/acled/Asia-Pacific_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""acled""","""Asia-Pacific_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""xlsx""",9.07
"""data/wdi_downloader_package/data/processed/world_bank_wdi_panel_long.csv""","""wdi_downloader_package/data/processed""","""world_bank_wdi_panel_long.csv""","""csv""",8.0
"""data/acled/Latin-America-the-Caribbean_aggregated_data_up_to_week_of-2026-06-20.…","""acled""","""Latin-America-the-Caribbean_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""xlsx""",7.56
"""data/UNHCR_RefugeeDataFinder_Copyright/persons_of_concern.csv""","""UNHCR_RefugeeDataFinder_Copyright""","""persons_of_concern.csv""","""csv""",6.82
"""data/acled/Middle-East_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""acled""","""Middle-East_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""xlsx""",6.31
"""data/acled/Europe-Central-Asia_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""acled""","""Europe-Central-Asia_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""xlsx""",5.15


In [6]:
largest = top_files.row(0, named=True)
dominant_ext = file_summary.row(0, named=True)
display_note(
    f"<strong>Execution insight.</strong> The inventory contains <b>{file_inventory.height}</b> files "
    f"for <b>{file_inventory['size_mb'].sum():,.2f} MB</b>. The dominant extension by storage volume is "
    f"<code>{dominant_ext['extension']}</code> ({dominant_ext['total_size_mb']:,.2f} MB across {dominant_ext['n_files']} files). "
    f"The largest file is <code>{largest['relative_path']}</code> ({largest['size_mb']:,.2f} MB)."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — global file inventory</strong>
<br><br>
<strong>Analytical question.</strong> What is the physical composition of the data archive before inspecting the analytical content?
<br>
<strong>Method.</strong> The notebook recursively scans <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">data/</code>, computes file extensions, file sizes and the largest files, then visualizes total volume by file type.
<br>
<strong>Observed result.</strong> The repository contains 63 files for approximately 504.91 MB. CSV files dominate total storage volume, and the largest file is <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">V-Dem-CY-Full+Others-v16.csv</code>, at approximately 387.53 MB.
<br>
<strong>Interpretation.</strong> The repository mixes analytical files, documentation and exported assets. File size mainly reflects processing cost, not analytical relevance. V-Dem requires a specific loading strategy because of its schema width.
<br>
<strong>EDA implication.</strong> Future work should distinguish analytical value from technical weight. Large files should be loaded selectively unless a specific analysis requires the full schema.
</div>

<a id="dataset-loading"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 4</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Dataset loading</h2>
</div>

Some files are loaded as compact profiling subsets to keep the notebook executable on a standard laptop. This is a loading strategy, not a dataset selection.

In [7]:
# UCDP GED is large and contains heavy source text columns. Load core EDA columns only, while keeping the full schema count.
ucdp_ged_path = DATA_DIR / "ucdp" / "ged261-csv.zip"
ucdp_ged_full_columns = zip_csv_header(ucdp_ged_path)
ucdp_ged_core_columns = [
    "id", "relid", "year", "type_of_violence", "conflict_new_id", "conflict_name",
    "dyad_new_id", "dyad_name", "side_a", "side_b", "country", "country_id", "region",
    "where_prec", "latitude", "longitude", "date_start", "date_end",
    "deaths_a", "deaths_b", "deaths_civilians", "deaths_unknown", "best", "high", "low",
]
ucdp_ged = read_csv_from_zip(ucdp_ged_path, columns=[c for c in ucdp_ged_core_columns if c in ucdp_ged_full_columns])

# Other UCDP tables are small enough to load fully.
ucdp_one_sided = read_csv_from_zip(DATA_DIR / "ucdp" / "ucdp-onesided-261-csv.zip")
ucdp_organized_cy = read_csv_from_zip(DATA_DIR / "ucdp" / "organizedviolencecy-261-csv.zip")
ucdp_actor = read_csv_from_zip(DATA_DIR / "ucdp" / "ucdp-actor-261-csv.zip")

# ACLED summary tables.
acled_country_year_events = read_excel_file(DATA_DIR / "acled" / "number_of_political_violence_events_by_country-year_as-of-26Jun2026.xlsx")
acled_country_month_events = read_excel_file(DATA_DIR / "acled" / "number_of_political_violence_events_by_country-month-year_as-of-26Jun2026.xlsx")
acled_civilian_targeting_cy = read_excel_file(DATA_DIR / "acled" / "number_of_events_targeting_civilians_by_country-year_as-of-26Jun2026.xlsx")
acled_civilian_fatalities_cy = read_excel_file(DATA_DIR / "acled" / "number_of_reported_civilian_fatalities_by_country-year_as-of-26Jun2026.xlsx")
acled_fatalities_cy = read_excel_file(DATA_DIR / "acled" / "number_of_reported_fatalities_by_country-year_as-of-26Jun2026.xlsx")

# Other data sources.
svac_complete = read_excel_file(DATA_DIR / "svac" / "SVAC_3.3_complete.xlsx")
svac_conflictyears = read_excel_file(DATA_DIR / "svac" / "SVAC_3.3_conflictyears.xlsx")
epr = read_csv_file(DATA_DIR / "EPR" / "EPR-2021.csv")
unhcr_poc = read_csv_file(DATA_DIR / "UNHCR_RefugeeDataFinder_Copyright" / "persons_of_concern.csv")
unhcr_demo = read_csv_file(DATA_DIR / "UNHCR_RefugeeDataFinder_Copyright - demo" / "persons_of_concern_demographics.csv")
wdi_wide = read_csv_file(DATA_DIR / "wdi_downloader_package" / "data" / "processed" / "world_bank_wdi_panel_wide.csv")
wdi_long = read_csv_file(DATA_DIR / "wdi_downloader_package" / "data" / "processed" / "world_bank_wdi_panel_long.csv")

# V-Dem is very wide. Load a core profiling subset, while recording the full header width.
vdem_path = DATA_DIR / "V-Dem-CY-FullOthers-v16_csv" / "V-Dem-CY-Full+Others-v16.csv"
with open(vdem_path, newline="", encoding="utf-8") as file:
    vdem_full_columns = next(csv.reader(file))
vdem_core_columns = [
    "country_name", "country_text_id", "country_id", "year", "COWcode",
    "v2x_polyarchy", "v2x_libdem", "v2x_partipdem", "v2x_delibdem", "v2x_egaldem",
]
vdem_core = pl.read_csv(
    vdem_path,
    columns=[col for col in vdem_core_columns if col in vdem_full_columns],
    infer_schema_length=1_000,
    ignore_errors=True,
    low_memory=True,
)

print("Loaded core analytical tables.")

Could not determine dtype for column 10, falling back to string


Could not determine dtype for column 10, falling back to string


Loaded core analytical tables.


In [8]:
# ACLED regional aggregate files are processed one by one to avoid keeping all regional rows in memory.
def process_acled_regional_files(paths: list[Path]) -> tuple[dict[str, Any], pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    profiles = []
    yearly = []
    country_year_keys = []
    null_totals: dict[str, int] = {}
    total_rows = 0
    coord_rows_total = 0
    countries = set()
    date_min = None
    date_max = None
    n_cols = None
    key_cols = ["WEEK", "COUNTRY", "ADMIN1", "REGION", "EVENTS", "FATALITIES", "CENTROID_LATITUDE", "CENTROID_LONGITUDE"]

    for path in paths:
        df = read_excel_file(path)
        total_rows += df.height
        n_cols = df.width if n_cols is None else n_cols
        region_label = path.name.replace("_aggregated_data_up_to_week_of-2026-06-20.xlsx", "")
        local_min, local_max = safe_min_max(df, "WEEK")
        date_min = local_min if date_min is None or local_min < date_min else date_min
        date_max = local_max if date_max is None or local_max > date_max else date_max
        countries.update(df.select(pl.col("COUNTRY").drop_nulls().unique())["COUNTRY"].to_list())
        coord_rows = df.select((pl.col("CENTROID_LATITUDE").is_not_null() & pl.col("CENTROID_LONGITUDE").is_not_null()).sum()).item()
        coord_rows_total += coord_rows
        for col in key_cols:
            if col in df.columns:
                null_totals[col] = null_totals.get(col, 0) + df.select(pl.col(col).is_null().sum()).item()
        yearly.append(
            df.with_columns(pl.col("WEEK").dt.year().alias("year"))
              .group_by(["year", "REGION"])
              .agg(pl.sum("EVENTS").alias("events"), pl.sum("FATALITIES").alias("fatalities"), pl.len().alias("rows"))
        )
        country_year_keys.append(
            df.select(
                pl.col("COUNTRY").cast(pl.String).map_elements(normalize_name, return_dtype=pl.String).alias("country_norm"),
                pl.col("WEEK").dt.year().cast(pl.Int64).alias("year"),
            ).drop_nulls(["country_norm", "year"]).unique().with_columns(pl.lit("ACLED regional aggregates").alias("dataset"))
        )
        profiles.append({"source_file": path.name, "region_label": region_label, "rows": df.height, "columns": df.width, "date_min": str(local_min), "date_max": str(local_max), "countries": df.select(pl.col("COUNTRY").n_unique()).item()})
        del df

    critical_null_max_pct = round(max(null_totals.values()) / total_rows * 100, 2) if null_totals else None
    profile = {
        "dataset": "ACLED regional aggregates",
        "family": "ACLED",
        "unit_of_analysis": "Admin-week aggregate",
        "n_rows": total_rows,
        "n_cols_loaded": n_cols,
        "n_cols_full_schema": n_cols,
        "year_min": date_min.year if date_min else None,
        "year_max": date_max.year if date_max else None,
        "n_years_observed": (date_max.year - date_min.year + 1) if date_min and date_max else None,
        "date_min": str(date_min) if date_min else None,
        "date_max": str(date_max) if date_max else None,
        "country_column": "COUNTRY",
        "n_country_values": len(countries),
        "region_column": "REGION",
        "n_region_values": len(paths),
        "has_coordinates": True,
        "coord_rows": coord_rows_total,
        "coord_coverage_pct": percent(coord_rows_total, total_rows),
        "critical_null_max_pct": critical_null_max_pct,
        "duplicate_rows_full_record": None,
        "identifier_columns_found": "ID, COUNTRY, ADMIN1, WEEK",
        "loading_note": "Processed file-by-file; full regional table not retained in memory.",
    }
    return profile, pl.concat(yearly).group_by(["year", "REGION"]).agg(pl.sum("events"), pl.sum("fatalities"), pl.sum("rows")).sort(["year", "REGION"]), pl.concat(country_year_keys).unique(), pl.DataFrame(profiles)

acled_regional_files = sorted((DATA_DIR / "acled").glob("*_aggregated_data_up_to_week_of-2026-06-20.xlsx"))
acled_regional_profile, acled_regional_yearly, acled_regional_cy_keys, acled_regional_file_profiles = process_acled_regional_files(acled_regional_files)

acled_regional_file_profiles

source_file,region_label,rows,columns,date_min,date_max,countries
str,str,i64,i64,str,str,i64
"""Africa_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""Africa""","276,103",13,"""1996-12-28""","""2026-06-20""",63
"""Asia-Pacific_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""Asia-Pacific""","214,387",13,"""2009-12-26""","""2026-06-20""",56
"""Europe-Central-Asia_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""Europe-Central-Asia""","123,330",13,"""2017-12-30""","""2026-06-20""",67
"""Latin-America-the-Caribbean_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""Latin-America-the-Caribbean""","178,894",13,"""2017-12-30""","""2026-06-20""",55
"""Middle-East_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""Middle-East""","149,543",13,"""2014-12-27""","""2026-06-20""",17
"""US-and-Canada_aggregated_data_up_to_week_of-2026-06-20.xlsx""","""US-and-Canada""","23,656",13,"""2018-11-10""","""2026-06-20""",10


In [9]:
loaded_shapes = pl.DataFrame([
    {"dataset": "UCDP GED core subset", "rows": ucdp_ged.height, "columns_loaded": ucdp_ged.width, "columns_full_schema": len(ucdp_ged_full_columns)},
    {"dataset": "UCDP One-sided", "rows": ucdp_one_sided.height, "columns_loaded": ucdp_one_sided.width, "columns_full_schema": ucdp_one_sided.width},
    {"dataset": "UCDP Organized Violence CY", "rows": ucdp_organized_cy.height, "columns_loaded": ucdp_organized_cy.width, "columns_full_schema": ucdp_organized_cy.width},
    {"dataset": "UCDP Actor", "rows": ucdp_actor.height, "columns_loaded": ucdp_actor.width, "columns_full_schema": ucdp_actor.width},
    {"dataset": "ACLED regional aggregates", "rows": acled_regional_profile["n_rows"], "columns_loaded": acled_regional_profile["n_cols_loaded"], "columns_full_schema": acled_regional_profile["n_cols_full_schema"]},
    {"dataset": "ACLED country-year events", "rows": acled_country_year_events.height, "columns_loaded": acled_country_year_events.width, "columns_full_schema": acled_country_year_events.width},
    {"dataset": "ACLED country-month events", "rows": acled_country_month_events.height, "columns_loaded": acled_country_month_events.width, "columns_full_schema": acled_country_month_events.width},
    {"dataset": "ACLED civilian targeting CY", "rows": acled_civilian_targeting_cy.height, "columns_loaded": acled_civilian_targeting_cy.width, "columns_full_schema": acled_civilian_targeting_cy.width},
    {"dataset": "ACLED civilian fatalities CY", "rows": acled_civilian_fatalities_cy.height, "columns_loaded": acled_civilian_fatalities_cy.width, "columns_full_schema": acled_civilian_fatalities_cy.width},
    {"dataset": "ACLED fatalities CY", "rows": acled_fatalities_cy.height, "columns_loaded": acled_fatalities_cy.width, "columns_full_schema": acled_fatalities_cy.width},
    {"dataset": "SVAC complete", "rows": svac_complete.height, "columns_loaded": svac_complete.width, "columns_full_schema": svac_complete.width},
    {"dataset": "SVAC conflict years", "rows": svac_conflictyears.height, "columns_loaded": svac_conflictyears.width, "columns_full_schema": svac_conflictyears.width},
    {"dataset": "EPR 2021", "rows": epr.height, "columns_loaded": epr.width, "columns_full_schema": epr.width},
    {"dataset": "UNHCR persons of concern", "rows": unhcr_poc.height, "columns_loaded": unhcr_poc.width, "columns_full_schema": unhcr_poc.width},
    {"dataset": "UNHCR demographics demo", "rows": unhcr_demo.height, "columns_loaded": unhcr_demo.width, "columns_full_schema": unhcr_demo.width},
    {"dataset": "WDI wide", "rows": wdi_wide.height, "columns_loaded": wdi_wide.width, "columns_full_schema": wdi_wide.width},
    {"dataset": "WDI long", "rows": wdi_long.height, "columns_loaded": wdi_long.width, "columns_full_schema": wdi_long.width},
    {"dataset": "V-Dem core subset", "rows": vdem_core.height, "columns_loaded": vdem_core.width, "columns_full_schema": len(vdem_full_columns)},
]).sort("rows", descending=True)

loaded_shapes

dataset,rows,columns_loaded,columns_full_schema
str,i64,i64,i64
"""ACLED regional aggregates""","965,913",13,13
"""UCDP GED core subset""","417,968",25,49
"""UNHCR persons of concern""","138,289",12,12
"""WDI long""","67,340",13,13
"""ACLED country-month events""","29,124",4,4
"""V-Dem core subset""","28,092",10,"4,618"
"""SVAC complete""","12,876",19,19
"""WDI wide""","9,620",11,11
"""SVAC conflict years""","7,545",19,19


In [10]:
total_loaded_rows = int(loaded_shapes["rows"].sum())
largest_loaded = loaded_shapes.sort("rows", descending=True).row(0, named=True)
widest_schema = loaded_shapes.sort("columns_full_schema", descending=True).row(0, named=True)
display_note(
    f"<strong>Execution insight.</strong> {loaded_shapes.height} tables are loaded or profiled, "
    f"representing <b>{total_loaded_rows:,}</b> observed rows in total. The largest table by row count is "
    f"<code>{largest_loaded['dataset']}</code> ({largest_loaded['rows']:,} rows). The widest schema is "
    f"<code>{widest_schema['dataset']}</code> ({widest_schema['columns_full_schema']:,} columns in the full schema). "
    "The partial loading of UCDP GED and V-Dem is a memory-stability choice, not an analytical selection decision."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — dataset loading</strong>
<br><br>
<strong>Analytical question.</strong> Can all relevant sources be profiled without turning this first notebook into a heavy processing pipeline?
<br>
<strong>Method.</strong> Small and medium-sized tables are loaded in full. UCDP GED is loaded with a selected set of core EDA columns while preserving the full schema count. V-Dem is profiled through a small set of core columns instead of loading all 4,618 variables. ACLED regional aggregates are processed file by file to avoid unnecessary memory pressure.
<br>
<strong>Observed result.</strong> The notebook loads or profiles 18 analytical tables, representing 1,709,079 observed rows. The largest loaded/profiled table is ACLED regional aggregates, with 965,913 rows. The widest full schema is V-Dem core subset, with 4,618 columns.
<br>
<strong>Interpretation.</strong> This loading strategy provides real structural diagnostics while avoiding unstable or unnecessary full-table reads. Partial loading is a reproducibility and memory-management decision, not an analytical source-selection decision.
<br>
<strong>EDA implication.</strong> If a later analysis focuses on a specific source, a dedicated full loading strategy may be required. This notebook remains a foundation-level profiling notebook.
</div>

<a id="technical-profiling"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 5</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Technical profiling</h2>
</div>

In [11]:
dataset_registry = [
    {"key":"ucdp_ged", "dataset":"UCDP GED core subset", "family":"UCDP", "unit_of_analysis":"Event", "df":ucdp_ged, "year_col":"year", "date_col":"date_start", "country_col":"country", "region_col":"region", "lat_col":"latitude", "lon_col":"longitude", "id_cols":["id","relid","conflict_new_id","dyad_new_id"], "n_cols_full_schema":len(ucdp_ged_full_columns), "loading_note":"Core EDA columns loaded; full source-text fields not loaded."},
    {"key":"ucdp_one_sided", "dataset":"UCDP One-sided", "family":"UCDP", "unit_of_analysis":"Actor-year", "df":ucdp_one_sided, "year_col":"year", "date_col":None, "country_col":"location", "region_col":"region", "lat_col":None, "lon_col":None, "id_cols":["conflict_id","dyad_id","actor_id"], "n_cols_full_schema":ucdp_one_sided.width, "loading_note":"Full table loaded."},
    {"key":"ucdp_organized_cy", "dataset":"UCDP Organized Violence CY", "family":"UCDP", "unit_of_analysis":"Country-year", "df":ucdp_organized_cy, "year_col":"year", "date_col":None, "country_col":"country", "region_col":"region", "lat_col":None, "lon_col":None, "id_cols":["country","country_id","year"], "n_cols_full_schema":ucdp_organized_cy.width, "loading_note":"Full table loaded."},
    {"key":"ucdp_actor", "dataset":"UCDP Actor", "family":"UCDP", "unit_of_analysis":"Actor metadata", "df":ucdp_actor, "year_col":None, "date_col":None, "country_col":"Location", "region_col":"Region", "lat_col":None, "lon_col":None, "id_cols":["ActorId","ConflictId","DyadId"], "n_cols_full_schema":ucdp_actor.width, "loading_note":"Full table loaded with utf8-lossy fallback if required."},
    {"key":"acled_country_year_events", "dataset":"ACLED country-year events", "family":"ACLED", "unit_of_analysis":"Country-year", "df":acled_country_year_events, "year_col":"YEAR", "date_col":None, "country_col":"COUNTRY", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["COUNTRY","YEAR"], "n_cols_full_schema":acled_country_year_events.width, "loading_note":"Full table loaded."},
    {"key":"acled_country_month_events", "dataset":"ACLED country-month events", "family":"ACLED", "unit_of_analysis":"Country-month", "df":acled_country_month_events, "year_col":"YEAR", "date_col":None, "country_col":"COUNTRY", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["COUNTRY","YEAR","MONTH"], "n_cols_full_schema":acled_country_month_events.width, "loading_note":"Full table loaded."},
    {"key":"acled_civilian_targeting_cy", "dataset":"ACLED civilian targeting CY", "family":"ACLED", "unit_of_analysis":"Country-year", "df":acled_civilian_targeting_cy, "year_col":"YEAR", "date_col":None, "country_col":"COUNTRY", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["COUNTRY","YEAR"], "n_cols_full_schema":acled_civilian_targeting_cy.width, "loading_note":"Full table loaded."},
    {"key":"acled_civilian_fatalities_cy", "dataset":"ACLED civilian fatalities CY", "family":"ACLED", "unit_of_analysis":"Country-year", "df":acled_civilian_fatalities_cy, "year_col":"YEAR", "date_col":None, "country_col":"COUNTRY", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["COUNTRY","YEAR"], "n_cols_full_schema":acled_civilian_fatalities_cy.width, "loading_note":"Full table loaded."},
    {"key":"acled_fatalities_cy", "dataset":"ACLED fatalities CY", "family":"ACLED", "unit_of_analysis":"Country-year", "df":acled_fatalities_cy, "year_col":"YEAR", "date_col":None, "country_col":"COUNTRY", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["COUNTRY","YEAR"], "n_cols_full_schema":acled_fatalities_cy.width, "loading_note":"Full table loaded."},
    {"key":"svac_complete", "dataset":"SVAC complete", "family":"SVAC", "unit_of_analysis":"Actor-year / conflict-year coding", "df":svac_complete, "year_col":"year", "date_col":None, "country_col":"location", "region_col":"region", "lat_col":None, "lon_col":None, "id_cols":["conflictid","actorid","year"], "n_cols_full_schema":svac_complete.width, "loading_note":"Full table loaded."},
    {"key":"svac_conflictyears", "dataset":"SVAC conflict years", "family":"SVAC", "unit_of_analysis":"Actor-year / conflict-year coding", "df":svac_conflictyears, "year_col":"year", "date_col":None, "country_col":"location", "region_col":"region", "lat_col":None, "lon_col":None, "id_cols":["conflictid","actorid","year"], "n_cols_full_schema":svac_conflictyears.width, "loading_note":"Full table loaded."},
    {"key":"epr", "dataset":"EPR 2021", "family":"EPR", "unit_of_analysis":"Group-period", "df":epr, "year_col":None, "date_col":None, "country_col":"statename", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["gwid","groupid","from","to"], "n_cols_full_schema":epr.width, "loading_note":"Full table loaded."},
    {"key":"unhcr_poc", "dataset":"UNHCR persons of concern", "family":"UNHCR", "unit_of_analysis":"Origin-asylum-year", "df":unhcr_poc, "year_col":"Year", "date_col":None, "country_col":"Country of Asylum", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["Country of Asylum ISO","Country of Origin ISO","Year"], "n_cols_full_schema":unhcr_poc.width, "loading_note":"Full table loaded."},
    {"key":"unhcr_demo", "dataset":"UNHCR demographics demo", "family":"UNHCR", "unit_of_analysis":"Origin-asylum-year-demographic group", "df":unhcr_demo, "year_col":"Year", "date_col":None, "country_col":"Country of Asylum", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["Country of Asylum ISO","Country of Origin ISO","Year"], "n_cols_full_schema":unhcr_demo.width, "loading_note":"Full table loaded."},
    {"key":"wdi_wide", "dataset":"WDI wide", "family":"World Bank", "unit_of_analysis":"Country-year", "df":wdi_wide, "year_col":"year", "date_col":None, "country_col":"country", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["iso3","year"], "n_cols_full_schema":wdi_wide.width, "loading_note":"Full table loaded."},
    {"key":"wdi_long", "dataset":"WDI long", "family":"World Bank", "unit_of_analysis":"Country-year-indicator", "df":wdi_long, "year_col":"year", "date_col":None, "country_col":"country", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["iso3","year","indicator_code"], "n_cols_full_schema":wdi_long.width, "loading_note":"Full table loaded."},
    {"key":"vdem_core", "dataset":"V-Dem core subset", "family":"V-Dem", "unit_of_analysis":"Country-year", "df":vdem_core, "year_col":"year", "date_col":None, "country_col":"country_name", "region_col":None, "lat_col":None, "lon_col":None, "id_cols":["country_text_id","country_id","COWcode","year"], "n_cols_full_schema":len(vdem_full_columns), "loading_note":"Core profiling columns loaded; full schema recorded."},
]

def profile_dataset(meta: dict[str, Any]) -> dict[str, Any]:
    df = meta["df"]
    year_col, date_col = meta.get("year_col"), meta.get("date_col")
    country_col, region_col = meta.get("country_col"), meta.get("region_col")
    lat_col, lon_col = meta.get("lat_col"), meta.get("lon_col")
    id_cols = [c for c in meta.get("id_cols", []) if c in df.columns]
    critical_cols = list(dict.fromkeys([c for c in [year_col, date_col, country_col, region_col, lat_col, lon_col, *id_cols] if c in df.columns]))
    year_min = year_max = None
    if year_col in df.columns:
        year_min, year_max = safe_min_max(df, year_col)
    elif date_col in df.columns:
        year_min = df.select(pl.col(date_col).dt.year().min()).item()
        year_max = df.select(pl.col(date_col).dt.year().max()).item()
    date_min = date_max = None
    if date_col in df.columns:
        date_min, date_max = safe_min_max(df, date_col)
    n_countries = df.select(pl.col(country_col).n_unique()).item() if country_col in df.columns else None
    n_regions = df.select(pl.col(region_col).n_unique()).item() if region_col in df.columns else None
    coord_rows = coord_pct = None
    has_coords = lat_col in df.columns and lon_col in df.columns
    if has_coords:
        coord_rows = df.select((pl.col(lat_col).is_not_null() & pl.col(lon_col).is_not_null()).sum()).item()
        coord_pct = percent(coord_rows, df.height)
    critical_null_max_pct = None
    if critical_cols:
        nulls = df.select([pl.col(c).is_null().sum().alias(c) for c in critical_cols]).to_dicts()[0]
        critical_null_max_pct = round(max(nulls.values()) / df.height * 100, 2) if df.height else None
    duplicate_rows = int(df.is_duplicated().sum()) if df.width <= 200 and df.height <= 1_000_000 else None
    return {
        "dataset": meta["dataset"], "family": meta["family"], "unit_of_analysis": meta["unit_of_analysis"],
        "n_rows": df.height, "n_cols_loaded": df.width, "n_cols_full_schema": meta.get("n_cols_full_schema", df.width),
        "year_min": year_min, "year_max": year_max,
        "n_years_observed": (int(year_max) - int(year_min) + 1) if isinstance(year_min, int) and isinstance(year_max, int) else None,
        "date_min": str(date_min) if date_min is not None else None, "date_max": str(date_max) if date_max is not None else None,
        "country_column": country_col if country_col in df.columns else None, "n_country_values": n_countries,
        "region_column": region_col if region_col in df.columns else None, "n_region_values": n_regions,
        "has_coordinates": has_coords, "coord_rows": coord_rows, "coord_coverage_pct": coord_pct,
        "critical_null_max_pct": critical_null_max_pct, "duplicate_rows_full_record": duplicate_rows,
        "identifier_columns_found": ", ".join(id_cols), "loading_note": meta.get("loading_note"),
    }

profiles = pl.concat([
    pl.DataFrame([profile_dataset(meta) for meta in dataset_registry]),
    pl.DataFrame([acled_regional_profile]),
], how="diagonal_relaxed").sort(["family", "dataset"])

profiles

dataset,family,unit_of_analysis,n_rows,n_cols_loaded,n_cols_full_schema,year_min,year_max,n_years_observed,date_min,date_max,country_column,n_country_values,region_column,n_region_values,has_coordinates,coord_rows,coord_coverage_pct,critical_null_max_pct,duplicate_rows_full_record,identifier_columns_found,loading_note
str,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,i64,bool,i64,f64,f64,i64,str,str
"""ACLED civilian fatalities CY""","""ACLED""","""Country-year""","2,717",3,3,"1,997","2,026",30,null,null,"""COUNTRY""",250,null,null,false,null,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED civilian targeting CY""","""ACLED""","""Country-year""","2,717",3,3,"1,997","2,026",30,null,null,"""COUNTRY""",250,null,null,false,null,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED country-month events""","""ACLED""","""Country-month""","29,124",4,4,"1,997","2,026",30,null,null,"""COUNTRY""",250,null,null,false,null,null,0.0,0,"""COUNTRY, YEAR, MONTH""","""Full table loaded."""
"""ACLED country-year events""","""ACLED""","""Country-year""","2,754",3,3,"1,997","2,026",30,null,null,"""COUNTRY""",250,null,null,false,null,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED fatalities CY""","""ACLED""","""Country-year""","2,983",3,3,"1,997","2,026",30,null,null,"""COUNTRY""",250,null,null,false,null,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED regional aggregates""","""ACLED""","""Admin-week aggregate""","965,913",13,13,"1,996","2,026",31,"""1996-12-28""","""2026-06-20""","""COUNTRY""",244,"""REGION""",6,true,"965,913",100.0,0.0,null,"""ID, COUNTRY, ADMIN1, WEEK""","""Processed file-by-file; full regional table not retained in memory."""
"""EPR 2021""","""EPR""","""Group-period""","4,339",11,11,null,null,null,null,null,"""statename""",181,null,null,false,null,null,0.0,0,"""gwid, groupid, from, to""","""Full table loaded."""
"""SVAC complete""","""SVAC""","""Actor-year / conflict-year coding""","12,876",19,19,"1,989","2,023",35,null,null,"""location""",110,"""region""",5,false,null,null,0.16,6,"""conflictid, actorid, year""","""Full table loaded."""
"""SVAC conflict years""","""SVAC""","""Actor-year / conflict-year coding""","7,545",19,19,"1,989","2,023",35,null,null,"""location""",110,"""region""",5,false,null,null,0.25,2,"""conflictid, actorid, year""","""Full table loaded."""


In [12]:
profiles_standard = profiles.filter(pl.col("family") != "V-Dem")
vdem_profile = profiles.filter(pl.col("family") == "V-Dem")

fig = px.scatter(
    to_pandas(profiles_standard),
    x="n_cols_full_schema",
    y="n_rows",
    size="n_rows",
    color="family",
    hover_name="dataset",
    hover_data=["unit_of_analysis", "n_cols_loaded", "n_cols_full_schema"],
    title="Dataset size profile excluding V-Dem schema-width outlier",
    labels={
        "n_cols_full_schema": "Number of columns in full schema",
        "n_rows": "Rows / observations",
    },
    log_y=True,
)
fig.update_layout(height=560)
show_plot(fig, "chart_03a_size_profile_excluding_vdem")

fig = px.bar(
    to_pandas(vdem_profile.select(["dataset", "n_cols_full_schema", "n_rows", "n_cols_loaded"])),
    x="dataset",
    y="n_cols_full_schema",
    text="n_cols_full_schema",
    hover_data=["n_rows", "n_cols_loaded"],
    title="V-Dem schema-width outlier",
    labels={
        "dataset": "Dataset",
        "n_cols_full_schema": "Number of columns in full schema",
    },
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(height=430, yaxis_range=[0, max(5000, int(vdem_profile.select(pl.col("n_cols_full_schema").max()).item() * 1.12))])
show_plot(fig, "chart_03b_vdem_schema_width_outlier")

vdem_cols = int(vdem_profile.select(pl.col("n_cols_full_schema").max()).item()) if vdem_profile.height else None
standard_max_cols = int(profiles_standard.select(pl.col("n_cols_full_schema").max()).item())
display_note(
    f"<strong>Execution insight.</strong> The size profile is split into two views: "
    f"all datasets excluding V-Dem, then V-Dem alone. This prevents the V-Dem schema width "
    f"({vdem_cols:,} columns) from compressing the X-axis scale for all other sources. "
    f"The widest full schema outside V-Dem contains {standard_max_cols:,} columns."
)


In [13]:
display(Markdown("### Largest tables by observed row count"))
display(profiles.sort("n_rows", descending=True).select(["dataset", "family", "unit_of_analysis", "n_rows", "n_cols_loaded", "n_cols_full_schema"]).head(8))

display(Markdown("### Widest schemas"))
display(profiles.sort("n_cols_full_schema", descending=True).select(["dataset", "family", "n_rows", "n_cols_loaded", "n_cols_full_schema"]).head(8))

largest_profile = profiles.sort("n_rows", descending=True).row(0, named=True)
widest_profile = profiles.sort("n_cols_full_schema", descending=True).row(0, named=True)
display_note(
    f"<strong>Execution insight.</strong> The profile highlights two different constraints: "
    f"<code>{largest_profile['dataset']}</code> dominates by row count ({largest_profile['n_rows']:,}), "
    f"whereas <code>{widest_profile['dataset']}</code> dominates by schema width "
    f"({widest_profile['n_cols_full_schema']:,} columns). This distinction informs profiling methods without selecting an analytical source."
)

### Largest tables by observed row count

dataset,family,unit_of_analysis,n_rows,n_cols_loaded,n_cols_full_schema
str,str,str,i64,i64,i64
"""ACLED regional aggregates""","""ACLED""","""Admin-week aggregate""","965,913",13,13
"""UCDP GED core subset""","""UCDP""","""Event""","417,968",25,49
"""UNHCR persons of concern""","""UNHCR""","""Origin-asylum-year""","138,289",12,12
"""WDI long""","""World Bank""","""Country-year-indicator""","67,340",13,13
"""ACLED country-month events""","""ACLED""","""Country-month""","29,124",4,4
"""V-Dem core subset""","""V-Dem""","""Country-year""","28,092",10,"4,618"
"""SVAC complete""","""SVAC""","""Actor-year / conflict-year coding""","12,876",19,19
"""WDI wide""","""World Bank""","""Country-year""","9,620",11,11


### Widest schemas

dataset,family,n_rows,n_cols_loaded,n_cols_full_schema
str,str,i64,i64,i64
"""V-Dem core subset""","""V-Dem""","28,092",10,"4,618"
"""UCDP Organized Violence CY""","""UCDP""","7,132",74,74
"""UCDP GED core subset""","""UCDP""","417,968",25,49
"""UCDP Actor""","""UCDP""","1,987",35,35
"""UNHCR demographics demo""","""UNHCR""","6,275",20,20
"""SVAC complete""","""SVAC""","12,876",19,19
"""SVAC conflict years""","""SVAC""","7,545",19,19
"""UCDP One-sided""","""UCDP""","1,408",17,17


<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — technical profiling</strong>
<br><br>
<strong>Analytical question.</strong> Which sources are large because they contain many observations, and which are large because they contain many variables?
<br>
<strong>Method.</strong> The notebook compares row counts, loaded column counts and full schema widths for each profiled table. V-Dem is shown separately as a schema-width outlier so it does not visually distort comparisons among the other sources.
<br>
<strong>Observed result.</strong> ACLED regional aggregates dominate by row count, with 965,913 rows. UCDP GED follows with 417,968 rows, and UNHCR persons of concern contains 138,289 rows. V-Dem is the main schema-width outlier, with 4,618 columns in the full source file and 10 columns loaded for profiling.
<br>
<strong>Interpretation.</strong> Row volume and schema width are two different technical constraints. Event or aggregate datasets may be heavy because they contain many observations, while V-Dem is heavy because it contains a very large number of variables.
<br>
<strong>EDA implication.</strong> Future profiling and visualization methods should treat V-Dem as a high-dimensional contextual source rather than as a standard table in schema-width comparisons.
</div>

<a id="temporal-coverage"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 6</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Temporal coverage</h2>
</div>

In [14]:
temporal_profiles = (
    profiles.filter(pl.col("year_min").is_not_null() & pl.col("year_max").is_not_null())
    .with_columns(pl.date(pl.col("year_min").cast(pl.Int32), 1, 1).alias("start"), pl.date(pl.col("year_max").cast(pl.Int32), 12, 31).alias("end"))
    .sort("year_min")
)

temporal_profiles.select(["dataset", "family", "unit_of_analysis", "year_min", "year_max", "n_years_observed"])

dataset,family,unit_of_analysis,year_min,year_max,n_years_observed
str,str,str,i64,i64,i64
"""V-Dem core subset""","""V-Dem""","""Country-year""","1,789","2,025",237
"""UNHCR persons of concern""","""UNHCR""","""Origin-asylum-year""","1,951","2,025",75
"""SVAC complete""","""SVAC""","""Actor-year / conflict-year coding""","1,989","2,023",35
"""SVAC conflict years""","""SVAC""","""Actor-year / conflict-year coding""","1,989","2,023",35
"""UCDP GED core subset""","""UCDP""","""Event""","1,989","2,025",37
"""UCDP One-sided""","""UCDP""","""Actor-year""","1,989","2,025",37
"""UCDP Organized Violence CY""","""UCDP""","""Country-year""","1,989","2,025",37
"""WDI long""","""World Bank""","""Country-year-indicator""","1,989","2,025",37
"""WDI wide""","""World Bank""","""Country-year""","1,989","2,025",37


In [15]:
fig = px.timeline(
    to_pandas(temporal_profiles),
    x_start="start",
    x_end="end",
    y="dataset",
    color="family",
    hover_data=["unit_of_analysis", "year_min", "year_max", "n_years_observed"],
    title="A — Temporal coverage by dataset: min year → max year",
)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=760)
show_plot(fig, "chart_04a_temporal_coverage_by_dataset")

In [16]:
def counts_by_year(meta: dict[str, Any]) -> pl.DataFrame | None:
    df = meta["df"]
    year_col, date_col = meta.get("year_col"), meta.get("date_col")
    if year_col in df.columns:
        frame = df.select(pl.col(year_col).cast(pl.Int64, strict=False).alias("year"))
    elif date_col in df.columns:
        frame = df.select(pl.col(date_col).dt.year().cast(pl.Int64).alias("year"))
    else:
        return None
    return frame.drop_nulls("year").group_by("year").agg(pl.len().alias("n_observations")).with_columns(pl.lit(meta["dataset"]).alias("dataset"), pl.lit(meta["family"]).alias("family"))

yearly_counts = pl.concat([out for meta in dataset_registry if (out := counts_by_year(meta)) is not None], how="diagonal_relaxed")
yearly_counts = pl.concat([
    yearly_counts,
    acled_regional_yearly.select(pl.col("year"), pl.col("rows").alias("n_observations")).group_by("year").agg(pl.sum("n_observations")).with_columns(pl.lit("ACLED regional aggregates").alias("dataset"), pl.lit("ACLED").alias("family"))
], how="diagonal_relaxed")

yearly_counts.head()

year,n_observations,dataset,family
i64,u32,str,str
"1,998","3,791","""UCDP GED core subset""","""UCDP"""
"2,008","6,790","""UCDP GED core subset""","""UCDP"""
"2,022","21,013","""UCDP GED core subset""","""UCDP"""
"2,010","9,139","""UCDP GED core subset""","""UCDP"""
"1,991","2,877","""UCDP GED core subset""","""UCDP"""


In [17]:
YEARLY_WINDOW_START = 1989
YEARLY_WINDOW_END = 2025

yearly_counts_window = yearly_counts.filter(
    (pl.col("year") >= YEARLY_WINDOW_START) & (pl.col("year") <= YEARLY_WINDOW_END)
)

dataset_order = (
    yearly_counts_window
    .select(["family", "dataset"])
    .unique()
    .sort(["family", "dataset"])
    .get_column("dataset")
    .to_list()
)
years_window = list(range(YEARLY_WINDOW_START, YEARLY_WINDOW_END + 1))

yearly_heatmap_grid = (
    pl.DataFrame({"dataset": dataset_order})
    .join(pl.DataFrame({"year": years_window}), how="cross")
)

yearly_heatmap = (
    yearly_heatmap_grid
    .join(
        yearly_counts_window.select(["dataset", "year", "n_observations"]),
        on=["dataset", "year"],
        how="left",
    )
    .with_columns(
        pl.col("n_observations").fill_null(0).cast(pl.Int64),
    )
    .with_columns(
        (pl.col("n_observations").cast(pl.Float64) + 1).log10().alias("log10_observations")
    )
)

heatmap_pd = to_pandas(yearly_heatmap)
z_log = (
    heatmap_pd
    .pivot(index="dataset", columns="year", values="log10_observations")
    .reindex(index=dataset_order, columns=years_window)
    .fillna(0)
)
z_raw = (
    heatmap_pd
    .pivot(index="dataset", columns="year", values="n_observations")
    .reindex(index=dataset_order, columns=years_window)
    .fillna(0)
    .astype(int)
)

import plotly.graph_objects as go

fig = go.Figure(
    data=go.Heatmap(
        z=z_log.values,
        x=years_window,
        y=dataset_order,
        customdata=z_raw.values,
        colorbar={"title": "log10(rows + 1)"},
        hovertemplate=(
            "Dataset=%{y}<br>"
            "Year=%{x}<br>"
            "Rows / observations=%{customdata:,}<br>"
            "log10(rows + 1)=%{z:.2f}"
            "<extra></extra>"
        ),
    )
)
fig.update_layout(
    title="B — Yearly observation density by dataset, restricted to 1989–2025",
    xaxis_title="Year",
    yaxis_title="Dataset",
    height=max(620, 34 * len(dataset_order) + 240),
    margin={"l": 230, "r": 40, "t": 80, "b": 60},
)
fig.update_xaxes(dtick=2)
show_plot(fig, "chart_04b_yearly_observation_density_heatmap_1989_2025")

display(Markdown("### Yearly observation density table used for the heatmap"))
yearly_counts_window.sort(["dataset", "year"]).head(20)

### Yearly observation density table used for the heatmap

year,n_observations,dataset,family
i64,u32,str,str
"1,997",48,"""ACLED civilian fatalities CY""","""ACLED"""
"1,998",48,"""ACLED civilian fatalities CY""","""ACLED"""
"1,999",48,"""ACLED civilian fatalities CY""","""ACLED"""
"2,000",48,"""ACLED civilian fatalities CY""","""ACLED"""
"2,001",48,"""ACLED civilian fatalities CY""","""ACLED"""
"2,002",48,"""ACLED civilian fatalities CY""","""ACLED"""
"2,003",48,"""ACLED civilian fatalities CY""","""ACLED"""
"2,004",48,"""ACLED civilian fatalities CY""","""ACLED"""
"2,005",48,"""ACLED civilian fatalities CY""","""ACLED"""


In [18]:
earliest = temporal_profiles.sort("year_min").row(0, named=True)
latest_year = int(temporal_profiles.select(pl.col("year_max").max()).item())
n_datasets_in_window = yearly_counts_window.select("dataset").n_unique()
max_yearly_point = yearly_counts_window.sort("n_observations", descending=True).row(0, named=True)
zero_cells = int(yearly_heatmap.filter(pl.col("n_observations") == 0).height)
total_heatmap_cells = yearly_heatmap.height
zero_share = zero_cells / total_heatmap_cells if total_heatmap_cells else 0

display_note(
    f"<strong>Execution insight.</strong> Chart A documents the complete observed temporal bounds: "
    f"the earliest coverage belongs to <code>{earliest['dataset']}</code> "
    f"({int(earliest['year_min'])}–{int(earliest['year_max'])}), and the latest year observed in at least one source is {latest_year}. "
    f"Chart B replaces unreadable line overlays with a <code>dataset × year</code> heatmap limited to "
    f"{YEARLY_WINDOW_START}–{YEARLY_WINDOW_END}. This view keeps {n_datasets_in_window} datasets and encodes "
    f"yearly density with <code>log10(rows + 1)</code>, preventing high-volume tables from visually overwhelming smaller tables. "
    f"Within this window, the largest yearly point is <code>{max_yearly_point['dataset']}</code> in {int(max_yearly_point['year'])} "
    f"with {int(max_yearly_point['n_observations']):,} observations. "
    f"{zero_share:.1%} of dataset-year cells are empty, making coverage gaps visible. "
    "Temporal coverage and yearly density therefore describe different properties of the sources."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — temporal coverage</strong>
<br><br>
<strong>Analytical question.</strong> Are the datasets comparable in terms of time coverage, or do they represent different historical windows?
<br>
<strong>Method.</strong> Temporal coverage is visualized in two complementary ways: a full min/max timeline by dataset, and a yearly density heatmap restricted to 1989–2025. The restricted view improves readability and avoids allowing long historical datasets to dominate the visual comparison.
<br>
<strong>Observed result.</strong> Temporal coverage is highly heterogeneous. V-Dem covers 1789–2025, UNHCR persons of concern covers 1951–2025, key UCDP and WDI sources cover 1989–2025, SVAC covers 1989–2023, and the observed ACLED exports extend to 2026. In the 1989–2025 heatmap, the largest yearly point is ACLED regional aggregates in 2025, with 107,872 observations. Around 14.7% of dataset-year cells in the heatmap are empty.
<br>
<strong>Interpretation.</strong> Temporal coverage and yearly observation density are not the same thing. Some datasets cover long periods with relatively stable or aggregated records, while others concentrate many observations in a shorter modern window.
<br>
<strong>EDA implication.</strong> Any future comparison must explicitly define its time window before comparing volumes, trends or source coverage.
</div>

<a id="geographic-coverage"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 7</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Geographic coverage and coordinates</h2>
</div>

In [19]:
geographic_profiles = profiles.filter(pl.col("n_country_values").is_not_null()).select([
    "dataset", "family", "unit_of_analysis", "country_column", "n_country_values", "has_coordinates", "coord_coverage_pct"
]).sort("n_country_values", descending=True)

geographic_profiles

dataset,family,unit_of_analysis,country_column,n_country_values,has_coordinates,coord_coverage_pct
str,str,str,str,i64,bool,f64
"""UCDP Actor""","""UCDP""","""Actor metadata""","""Location""",363,false,null
"""WDI long""","""World Bank""","""Country-year-indicator""","""country""",260,false,null
"""WDI wide""","""World Bank""","""Country-year""","""country""",260,false,null
"""ACLED civilian fatalities CY""","""ACLED""","""Country-year""","""COUNTRY""",250,false,null
"""ACLED civilian targeting CY""","""ACLED""","""Country-year""","""COUNTRY""",250,false,null
"""ACLED country-month events""","""ACLED""","""Country-month""","""COUNTRY""",250,false,null
"""ACLED country-year events""","""ACLED""","""Country-year""","""COUNTRY""",250,false,null
"""ACLED fatalities CY""","""ACLED""","""Country-year""","""COUNTRY""",250,false,null
"""ACLED regional aggregates""","""ACLED""","""Admin-week aggregate""","""COUNTRY""",244,true,100.0


In [20]:
fig = px.bar(
    to_pandas(geographic_profiles), x="n_country_values", y="dataset", color="family", orientation="h",
    title="Raw distinct country/location values by dataset",
    labels={"n_country_values":"Distinct raw country/location values", "dataset":"Dataset"},
    hover_data=["country_column", "unit_of_analysis", "has_coordinates", "coord_coverage_pct"],
)
fig.update_layout(height=760)
fig.update_yaxes(autorange="reversed")
show_plot(fig, "chart_06")

In [21]:
coordinate_profiles = profiles.filter(pl.col("has_coordinates") == True).select(["dataset", "family", "n_rows", "coord_rows", "coord_coverage_pct"])
display(coordinate_profiles)

if coordinate_profiles.height:
    coord_list = ", ".join(coordinate_profiles["dataset"].to_list())
    min_coord_cov = coordinate_profiles.select(pl.col("coord_coverage_pct").min()).item()
    display_note(
        f"<strong>Execution insight.</strong> The tables with explicit coordinates in this profiling are: {coord_list}. "
        f"The minimum coordinate coverage among them is {min_coord_cov:.2f}%. "
        "This indicates technical geographic usability, but it does not yet validate the semantic quality of geocoding."
    )

dataset,family,n_rows,coord_rows,coord_coverage_pct
str,str,i64,i64,f64
"""ACLED regional aggregates""","""ACLED""","965,913","965,913",100.0
"""UCDP GED core subset""","""UCDP""","417,968","417,968",100.0


<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — geographic coverage and coordinates</strong>
<br><br>
<strong>Analytical question.</strong> Which datasets contain usable geographic information, and at what level?
<br>
<strong>Method.</strong> The notebook counts distinct values in available country or location columns and identifies tables with explicit latitude/longitude fields.
<br>
<strong>Observed result.</strong> Raw geographic values vary substantially across sources. WDI contains 260 country or territory values, ACLED country-level summaries contain 250, V-Dem contains 202, and UCDP Organized Violence CY contains 199. Explicit coordinates are available in ACLED regional aggregates and UCDP GED core subset. In both cases, coordinate coverage is 100% for the loaded/profiled rows.
<br>
<strong>Interpretation.</strong> A distinct location value is not automatically a comparable country. It may represent a territory, historical entity, naming convention, multi-country location or source-specific label. Coordinate completeness also does not guarantee semantic geocoding quality.
<br>
<strong>EDA implication.</strong> Cross-source geographic analysis requires an explicit country identifier and harmonization strategy before interpretation.
</div>

<a id="granularity-identifiers"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 8</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Granularity and identifiers</h2>
</div>

In [22]:
granularity_summary = (
    profiles.group_by("unit_of_analysis")
    .agg(pl.len().alias("n_tables"), pl.sum("n_rows").alias("total_rows_observed"), pl.col("dataset").sort().str.join(", ").alias("datasets"))
    .sort("n_tables", descending=True)
)

granularity_summary

unit_of_analysis,n_tables,total_rows_observed,datasets
str,u32,i64,str
"""Country-year""",7,"56,015","""ACLED civilian fatalities CY, ACLED civilian targeting CY, ACLED country-year ev…"
"""Actor-year / conflict-year coding""",2,"20,421","""SVAC complete, SVAC conflict years"""
"""Group-period""",1,"4,339","""EPR 2021"""
"""Origin-asylum-year""",1,"138,289","""UNHCR persons of concern"""
"""Country-year-indicator""",1,"67,340","""WDI long"""
"""Admin-week aggregate""",1,"965,913","""ACLED regional aggregates"""
"""Event""",1,"417,968","""UCDP GED core subset"""
"""Country-month""",1,"29,124","""ACLED country-month events"""
"""Actor-year""",1,"1,408","""UCDP One-sided"""


In [23]:
identifier_matrix = profiles.select([
    "dataset", "family", "unit_of_analysis", "identifier_columns_found", "country_column", "year_min", "year_max", "has_coordinates", "loading_note"
])
identifier_matrix

dataset,family,unit_of_analysis,identifier_columns_found,country_column,year_min,year_max,has_coordinates,loading_note
str,str,str,str,str,i64,i64,bool,str
"""ACLED civilian fatalities CY""","""ACLED""","""Country-year""","""COUNTRY, YEAR""","""COUNTRY""","1,997","2,026",false,"""Full table loaded."""
"""ACLED civilian targeting CY""","""ACLED""","""Country-year""","""COUNTRY, YEAR""","""COUNTRY""","1,997","2,026",false,"""Full table loaded."""
"""ACLED country-month events""","""ACLED""","""Country-month""","""COUNTRY, YEAR, MONTH""","""COUNTRY""","1,997","2,026",false,"""Full table loaded."""
"""ACLED country-year events""","""ACLED""","""Country-year""","""COUNTRY, YEAR""","""COUNTRY""","1,997","2,026",false,"""Full table loaded."""
"""ACLED fatalities CY""","""ACLED""","""Country-year""","""COUNTRY, YEAR""","""COUNTRY""","1,997","2,026",false,"""Full table loaded."""
"""ACLED regional aggregates""","""ACLED""","""Admin-week aggregate""","""ID, COUNTRY, ADMIN1, WEEK""","""COUNTRY""","1,996","2,026",true,"""Processed file-by-file; full regional table not retained in memory."""
"""EPR 2021""","""EPR""","""Group-period""","""gwid, groupid, from, to""","""statename""",null,null,false,"""Full table loaded."""
"""SVAC complete""","""SVAC""","""Actor-year / conflict-year coding""","""conflictid, actorid, year""","""location""","1,989","2,023",false,"""Full table loaded."""
"""SVAC conflict years""","""SVAC""","""Actor-year / conflict-year coding""","""conflictid, actorid, year""","""location""","1,989","2,023",false,"""Full table loaded."""


In [24]:
fig = px.bar(
    to_pandas(granularity_summary), x="unit_of_analysis", y="n_tables", hover_data=["datasets", "total_rows_observed"],
    title="Loaded/profiled tables by unit of analysis",
    labels={"unit_of_analysis":"Unit of analysis", "n_tables":"Number of tables"},
)
fig.update_layout(height=520, xaxis_tickangle=-30)
show_plot(fig, "chart_07")

main_granularity = granularity_summary.sort("n_tables", descending=True).row(0, named=True)
display_note(
    f"<strong>Execution insight.</strong> The most frequent unit of analysis is "
    f"<code>{main_granularity['unit_of_analysis']}</code>, with {main_granularity['n_tables']} tables. "
    f"However, the inventory contains {granularity_summary.height} distinct units of analysis. "
    "This diversity prevents any direct merge without first choosing a reference unit of observation."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — granularity and identifiers</strong>
<br><br>
<strong>Analytical question.</strong> Do the sources describe the same statistical objects?
<br>
<strong>Method.</strong> The notebook groups tables by unit of analysis and lists the identifiers available in each source, such as event IDs, actor IDs, country-year keys, ISO3 codes, conflict IDs and dyad IDs.
<br>
<strong>Observed result.</strong> The inventory contains 11 distinct units of analysis. Country-year is the most frequent granularity, with 7 tables, but the repository also includes event-level data, country-month data, actor-year records, group-period records, admin-week aggregates and origin-asylum-year tables.
<br>
<strong>Interpretation.</strong> The datasets should not be treated as directly stackable. An event-level table, an actor-year table and a country-year panel do not answer the same analytical question.
<br>
<strong>EDA implication.</strong> Any cross-source analysis must explicitly define its unit of observation before producing joins, aggregations or visual comparisons.
</div>

<a id="data-quality"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 9</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Data quality checks</h2>
</div>

In [25]:
quality_summary = profiles.select([
    "dataset", "family", "unit_of_analysis", "critical_null_max_pct", "duplicate_rows_full_record", "has_coordinates", "coord_coverage_pct"
]).sort("critical_null_max_pct", descending=True, nulls_last=True)

quality_summary

dataset,family,unit_of_analysis,critical_null_max_pct,duplicate_rows_full_record,has_coordinates,coord_coverage_pct
str,str,str,f64,i64,bool,f64
"""UCDP Actor""","""UCDP""","""Actor metadata""",61.75,59,false,null
"""V-Dem core subset""","""V-Dem""","""Country-year""",3.03,0,false,null
"""SVAC conflict years""","""SVAC""","""Actor-year / conflict-year coding""",0.25,2,false,null
"""SVAC complete""","""SVAC""","""Actor-year / conflict-year coding""",0.16,6,false,null
"""ACLED civilian fatalities CY""","""ACLED""","""Country-year""",0.0,0,false,null
"""ACLED civilian targeting CY""","""ACLED""","""Country-year""",0.0,0,false,null
"""ACLED country-month events""","""ACLED""","""Country-month""",0.0,0,false,null
"""ACLED country-year events""","""ACLED""","""Country-year""",0.0,0,false,null
"""ACLED fatalities CY""","""ACLED""","""Country-year""",0.0,0,false,null


In [26]:
fig = px.bar(
    to_pandas(quality_summary.filter(pl.col("critical_null_max_pct").is_not_null())),
    x="critical_null_max_pct", y="dataset", color="family", orientation="h",
    title="Maximum missingness among declared critical columns",
    labels={"critical_null_max_pct":"Max null rate among critical columns (%)", "dataset":"Dataset"},
)
fig.update_layout(height=760)
fig.update_yaxes(autorange="reversed")
show_plot(fig, "chart_08")

In [27]:
def column_null_profile(meta: dict[str, Any], top_n: int = 8) -> pl.DataFrame:
    df = meta["df"]
    out = df.select([pl.col(col).is_null().sum().alias(col) for col in df.columns]).unpivot(variable_name="column", value_name="n_null")
    return out.with_columns(
        pl.lit(meta["dataset"]).alias("dataset"),
        (pl.col("n_null") / df.height * 100).round(2).alias("null_pct"),
    ).sort("null_pct", descending=True).head(top_n)

null_profiles = pl.concat([column_null_profile(meta) for meta in dataset_registry], how="diagonal_relaxed")
null_profiles.sort(["dataset", "null_pct"], descending=[False, True]).head(80)

column,n_null,dataset,null_pct
str,u32,str,f64
"""COUNTRY""",0,"""ACLED civilian fatalities CY""",0.0
"""FATALITIES""",0,"""ACLED civilian fatalities CY""",0.0
"""YEAR""",0,"""ACLED civilian fatalities CY""",0.0
"""COUNTRY""",0,"""ACLED civilian targeting CY""",0.0
"""YEAR""",0,"""ACLED civilian targeting CY""",0.0
"""EVENTS""",0,"""ACLED civilian targeting CY""",0.0
"""COUNTRY""",0,"""ACLED country-month events""",0.0
"""MONTH""",0,"""ACLED country-month events""",0.0
"""YEAR""",0,"""ACLED country-month events""",0.0


In [28]:
worst_quality = quality_summary.filter(pl.col("critical_null_max_pct").is_not_null()).sort("critical_null_max_pct", descending=True).row(0, named=True)
zero_critical = quality_summary.filter(pl.col("critical_null_max_pct") == 0).height
display_note(
    f"<strong>Execution insight.</strong> The highest observed critical missingness rate is "
    f"{worst_quality['critical_null_max_pct']:.2f}% for <code>{worst_quality['dataset']}</code>. "
    f"{zero_critical} tables have 0% missingness on the declared critical columns. "
    "These figures are triage signals: they should be read together with codebooks, not as final data quality verdicts."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — data quality checks</strong>
<br><br>
<strong>Analytical question.</strong> Which initial quality signals should be inspected before moving toward more interpretive analysis?
<br>
<strong>Method.</strong> The notebook computes missingness rates on declared critical columns, exact duplicate rows when feasible, and coordinate coverage for geocoded sources.
<br>
<strong>Observed result.</strong> Most profiled tables show low missingness on the selected critical columns. Fourteen tables have 0% missingness on declared critical columns. The highest observed critical missingness is UCDP Actor, at 61.75%. V-Dem core subset shows 3.03% missingness on the loaded critical columns, while SVAC remains below 0.3%.
<br>
<strong>Interpretation.</strong> These metrics are triage signals, not final quality verdicts. High missingness can reflect metadata structure, source design or optional fields. Low missingness does not guarantee that a variable is conceptually comparable across sources.
<br>
<strong>EDA implication.</strong> The next analytical step should combine these diagnostics with codebook review before using variables as substantive indicators.
</div>

<a id="first-descriptive-exploration"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 10</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">First descriptive exploration</h2>
</div>

These charts are first-pass diagnostic views. They check structure, scale and availability before any substantive interpretation.

In [29]:
ucdp_ged_year_type = (
    ucdp_ged.group_by(["year", "type_of_violence"])
    .agg(pl.len().alias("n_events"), pl.sum("best").alias("best_fatalities"))
    .sort(["year", "type_of_violence"])
)

fig = px.line(
    to_pandas(ucdp_ged_year_type), x="year", y="n_events", color="type_of_violence",
    title="UCDP GED — event count by year and type_of_violence",
    labels={"n_events":"Event rows", "year":"Year", "type_of_violence":"Type of violence"},
)
fig.update_layout(height=520)
show_plot(fig, "chart_09")

ucdp_ged_year_type.head()

year,type_of_violence,n_events,best_fatalities
i64,i64,u32,i64
"1,989",1,"1,761","55,777"
"1,989",2,314,"4,225"
"1,989",3,584,"8,309"
"1,990",1,"1,659","80,395"
"1,990",2,620,"5,287"


In [30]:
fig = px.line(
    to_pandas(acled_regional_yearly), x="year", y="events", color="REGION",
    title="ACLED regional aggregates — summed EVENTS by year and region",
    labels={"events":"Summed EVENTS", "year":"Year", "REGION":"Region"},
)
fig.update_layout(height=520)
show_plot(fig, "chart_10")

In [31]:
unhcr_value_cols = ["Refugees", "Asylum-seekers", "IDPs", "Other people in need of international protection", "Stateless persons", "Host community", "Others of concern"]
unhcr_yearly = (
    unhcr_poc.with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in unhcr_value_cols if c in unhcr_poc.columns])
    .group_by("Year").agg([pl.sum(c).alias(c) for c in unhcr_value_cols if c in unhcr_poc.columns]).sort("Year")
)
unhcr_yearly_long = unhcr_yearly.unpivot(index="Year", variable_name="category", value_name="value")

fig = px.line(
    to_pandas(unhcr_yearly_long), x="Year", y="value", color="category",
    title="UNHCR persons of concern — summed categories by year",
    labels={"value":"Summed value", "Year":"Year", "category":"Category"},
)
fig.update_layout(height=560)
show_plot(fig, "chart_11")

In [32]:
wdi_indicator_cols = [c for c in wdi_wide.columns if c not in {"country", "country_id", "iso3", "year"}]
wdi_missingness = pl.DataFrame([
    {"indicator": c, "non_null_rows": wdi_wide.select(pl.col(c).is_not_null().sum()).item(), "null_pct": round(wdi_wide.select(pl.col(c).is_null().sum()).item() / wdi_wide.height * 100, 2)}
    for c in wdi_indicator_cols
]).sort("null_pct", descending=True)

fig = px.bar(
    to_pandas(wdi_missingness), x="indicator", y="null_pct",
    title="WDI wide panel — missingness by selected indicator",
    labels={"indicator":"Indicator", "null_pct":"Null values (%)"},
)
fig.update_layout(height=500, xaxis_tickangle=-35)
show_plot(fig, "chart_12")

wdi_missingness

indicator,non_null_rows,null_pct
str,i64,f64
"""gini_index""","2,264",76.47
"""secondary_school_enrollment_gross_pct""","5,824",39.46
"""internet_users_pct_population""","6,812",29.19
"""under5_mortality_rate""","8,561",11.01
"""gdp_per_capita_constant_2015_usd""","9,052",5.9
"""population_total""","9,619",0.01
"""urban_population_pct_total""","9,620",0.0


In [33]:
worst_wdi = wdi_missingness.row(0, named=True)
best_wdi = wdi_missingness.sort("null_pct").row(0, named=True)
display_note(
    f"<strong>Execution insight.</strong> The descriptive visualizations are generated under <code>outputs/plotly_html/</code>. "
    f"In the WDI wide panel, the most incomplete indicator is <code>{worst_wdi['indicator']}</code> "
    f"({worst_wdi['null_pct']:.2f}% null values), while the most complete indicator is <code>{best_wdi['indicator']}</code> "
    f"({best_wdi['null_pct']:.2f}% null values). The previous charts are consistency checks, not substantive conclusions."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — first descriptive exploration</strong>
<br><br>
<strong>Analytical question.</strong> Do the loaded datasets produce readable time series and distributions for an initial consistency check?
<br>
<strong>Method.</strong> The notebook generates simple Plotly aggregations: UCDP GED event counts by year and type of violence, ACLED regional aggregates by year and region, UNHCR persons of concern by category and year, and WDI indicator missingness.
<br>
<strong>Observed result.</strong> The charts are generated and exported as HTML files under <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">outputs/plotly_html/</code>. The WDI wide panel shows substantial differences in indicator availability. The most incomplete selected WDI indicator is <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">gini_index</code>, with 76.47% null values, while <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">urban_population_pct_total</code> is complete in the loaded panel.
<br>
<strong>Interpretation.</strong> These visualizations are consistency checks. They help verify structure, order of magnitude and temporal behavior, but they should not be read as substantive conclusions about conflict dynamics.
<br>
<strong>EDA implication.</strong> Directly comparing curve heights across data producers would be fragile without harmonizing definitions, units and coverage first.
</div>

<a id="join-feasibility"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 11</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Join feasibility diagnostics</h2>
</div>

This section checks raw-normalized country-year overlap. It is a diagnostic only and does not replace proper country-code harmonization.

In [34]:
def country_year_keys(meta: dict[str, Any]) -> pl.DataFrame | None:
    df = meta["df"]
    year_col, country_col = meta.get("year_col"), meta.get("country_col")
    if year_col not in df.columns or country_col not in df.columns:
        return None
    return df.select(
        pl.col(country_col).cast(pl.String).map_elements(normalize_name, return_dtype=pl.String).alias("country_norm"),
        pl.col(year_col).cast(pl.Int64, strict=False).alias("year"),
    ).drop_nulls(["country_norm", "year"]).unique().with_columns(pl.lit(meta["dataset"]).alias("dataset"))

cy_key_frames = [out for meta in dataset_registry if (out := country_year_keys(meta)) is not None]
cy_keys = pl.concat([*cy_key_frames, acled_regional_cy_keys], how="diagonal_relaxed")

cy_key_summary = cy_keys.group_by("dataset").agg(
    pl.len().alias("n_country_year_keys"),
    pl.col("country_norm").n_unique().alias("n_country_names"),
    pl.col("year").min().alias("year_min"),
    pl.col("year").max().alias("year_max"),
).sort("n_country_year_keys", descending=True)

cy_key_summary

dataset,n_country_year_keys,n_country_names,year_min,year_max
str,u32,u32,i64,i64
"""V-Dem core subset""","28,092",202,"1,789","2,025"
"""WDI long""","9,620",260,"1,989","2,025"
"""WDI wide""","9,620",260,"1,989","2,025"
"""UNHCR persons of concern""","7,760",195,"1,951","2,025"
"""UCDP Organized Violence CY""","7,132",199,"1,989","2,025"
"""ACLED regional aggregates""","3,057",244,"1,996","2,026"
"""ACLED fatalities CY""","2,983",250,"1,997","2,026"
"""ACLED country-year events""","2,754",250,"1,997","2,026"
"""ACLED country-month events""","2,754",250,"1,997","2,026"


In [35]:
datasets_with_cy = cy_key_summary["dataset"].to_list()
key_sets = {
    d: set(cy_keys.filter(pl.col("dataset") == d).select(pl.concat_str(["country_norm", pl.col("year").cast(pl.String)], separator="|").alias("key"))["key"].to_list())
    for d in datasets_with_cy
}
rows = []
for left in datasets_with_cy:
    for right in datasets_with_cy:
        lset, rset = key_sets[left], key_sets[right]
        overlap, union = len(lset & rset), len(lset | rset)
        rows.append({"left_dataset": left, "right_dataset": right, "overlap_keys": overlap, "left_keys": len(lset), "right_keys": len(rset), "jaccard_pct": round(overlap / union * 100, 2) if union else None})
cy_overlap = pl.DataFrame(rows)
cy_overlap.head()

left_dataset,right_dataset,overlap_keys,left_keys,right_keys,jaccard_pct
str,str,i64,i64,i64,f64
"""V-Dem core subset""","""V-Dem core subset""","28,092","28,092","28,092",100.0
"""V-Dem core subset""","""WDI long""","5,563","28,092","9,620",17.3
"""V-Dem core subset""","""WDI wide""","5,563","28,092","9,620",17.3
"""V-Dem core subset""","""UNHCR persons of concern""","6,371","28,092","7,760",21.61
"""V-Dem core subset""","""UCDP Organized Violence CY""","5,745","28,092","7,132",19.49


In [36]:
heatmap_df = cy_overlap.pivot(index="left_dataset", columns="right_dataset", values="jaccard_pct")
fig = px.imshow(
    to_pandas(heatmap_df), text_auto=True, aspect="auto",
    title="Pairwise overlap of raw-normalized country-year keys — Jaccard %",
    labels={"x":"Right dataset", "y":"Left dataset", "color":"Jaccard %"},
)
fig.update_layout(height=820)
show_plot(fig, "chart_13")

In [37]:
non_self_overlap = cy_overlap.filter(pl.col("left_dataset") != pl.col("right_dataset"))
best_overlap = non_self_overlap.sort("jaccard_pct", descending=True).row(0, named=True)
worst_overlap = non_self_overlap.sort("jaccard_pct").row(0, named=True)
display_note(
    f"<strong>Execution insight.</strong> The strongest raw country-year overlap outside the diagonal is "
    f"<code>{best_overlap['left_dataset']}</code> ↔ <code>{best_overlap['right_dataset']}</code> "
    f"({best_overlap['jaccard_pct']:.2f}% Jaccard). The weakest overlap in this matrix is "
    f"<code>{worst_overlap['left_dataset']}</code> ↔ <code>{worst_overlap['right_dataset']}</code> "
    f"({worst_overlap['jaccard_pct']:.2f}%). These values reflect naming conventions as well as real coverage."
)

<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — join feasibility diagnostics</strong>
<br><br>
<strong>Analytical question.</strong> Do raw country-year keys overlap enough to support technical joins across sources?
<br>
<strong>Method.</strong> The notebook builds raw-normalized <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">country + year</code> keys and computes pairwise Jaccard overlap among datasets with both country and year fields.
<br>
<strong>Observed result.</strong> The overlap matrix is produced successfully. The strongest off-diagonal overlap is WDI wide ↔ WDI long, with 100.00% Jaccard overlap. The weakest observed overlap in the matrix is SVAC complete ↔ UNHCR demographics demo, with 0.00% Jaccard overlap.
<br>
<strong>Interpretation.</strong> These values reflect both real coverage and naming conventions. A low raw overlap does not necessarily mean that a join is impossible; it often means that explicit country-code harmonization is required.
<br>
<strong>EDA implication.</strong> This section should be treated as a technical diagnostic, not as analytical validation of cross-source joins.
</div>

<a id="comparative-eda-matrix"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 12</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Comparative EDA matrix</h2>
</div>

This is not a scorecard. It consolidates observed technical indicators.

In [38]:
comparative_eda_matrix = profiles.select([
    "dataset", "family", "unit_of_analysis", "n_rows", "n_cols_loaded", "n_cols_full_schema",
    "year_min", "year_max", "n_years_observed", "country_column", "n_country_values",
    "has_coordinates", "coord_coverage_pct", "critical_null_max_pct", "duplicate_rows_full_record",
    "identifier_columns_found", "loading_note",
]).sort(["family", "dataset"])

comparative_eda_matrix

dataset,family,unit_of_analysis,n_rows,n_cols_loaded,n_cols_full_schema,year_min,year_max,n_years_observed,country_column,n_country_values,has_coordinates,coord_coverage_pct,critical_null_max_pct,duplicate_rows_full_record,identifier_columns_found,loading_note
str,str,str,i64,i64,i64,i64,i64,i64,str,i64,bool,f64,f64,i64,str,str
"""ACLED civilian fatalities CY""","""ACLED""","""Country-year""","2,717",3,3,"1,997","2,026",30,"""COUNTRY""",250,false,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED civilian targeting CY""","""ACLED""","""Country-year""","2,717",3,3,"1,997","2,026",30,"""COUNTRY""",250,false,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED country-month events""","""ACLED""","""Country-month""","29,124",4,4,"1,997","2,026",30,"""COUNTRY""",250,false,null,0.0,0,"""COUNTRY, YEAR, MONTH""","""Full table loaded."""
"""ACLED country-year events""","""ACLED""","""Country-year""","2,754",3,3,"1,997","2,026",30,"""COUNTRY""",250,false,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED fatalities CY""","""ACLED""","""Country-year""","2,983",3,3,"1,997","2,026",30,"""COUNTRY""",250,false,null,0.0,0,"""COUNTRY, YEAR""","""Full table loaded."""
"""ACLED regional aggregates""","""ACLED""","""Admin-week aggregate""","965,913",13,13,"1,996","2,026",31,"""COUNTRY""",244,true,100.0,0.0,null,"""ID, COUNTRY, ADMIN1, WEEK""","""Processed file-by-file; full regional table not retained in memory."""
"""EPR 2021""","""EPR""","""Group-period""","4,339",11,11,null,null,null,"""statename""",181,false,null,0.0,0,"""gwid, groupid, from, to""","""Full table loaded."""
"""SVAC complete""","""SVAC""","""Actor-year / conflict-year coding""","12,876",19,19,"1,989","2,023",35,"""location""",110,false,null,0.16,6,"""conflictid, actorid, year""","""Full table loaded."""
"""SVAC conflict years""","""SVAC""","""Actor-year / conflict-year coding""","7,545",19,19,"1,989","2,023",35,"""location""",110,false,null,0.25,2,"""conflictid, actorid, year""","""Full table loaded."""


In [39]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
comparative_matrix_path = OUTPUT_DIR / "01_comparative_eda_matrix.csv"
comparative_eda_matrix.write_csv(comparative_matrix_path)
print(f"Comparative EDA matrix written to: {comparative_matrix_path.relative_to(PROJECT_ROOT)}")

display_note(
    f"<strong>Execution insight.</strong> The exported matrix contains {comparative_eda_matrix.height} rows and "
    f"{comparative_eda_matrix.width} descriptive columns. It remains deliberately factual: no score, no selection status, no V1 recommendation."
)

Comparative EDA matrix written to: outputs/01_comparative_eda_matrix.csv


<div class="cl-responsive cl-analytical-note" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:18px 0 28px 0;padding:16px 18px;border-left:4px solid #D4A44C;border-radius:8px;background:#F7F4EC;line-height:1.55;">
<strong>Analytical note — comparative EDA matrix</strong>
<br><br>
<strong>Analytical question.</strong> How can the observed findings be consolidated without turning exploratory profiling into premature dataset selection?
<br>
<strong>Method.</strong> The final matrix aggregates descriptive indicators only: source family, unit of analysis, row count, column count, temporal coverage, country values, coordinate availability, critical missingness, duplicates and available identifiers.
<br>
<strong>Observed result.</strong> The matrix contains 18 profiled tables and 17 descriptive columns. It is exported to <code style="white-space:normal;overflow-wrap:anywhere;word-break:break-word;">outputs/01_comparative_eda_matrix.csv</code>.
<br>
<strong>Interpretation.</strong> The matrix is a factual framing tool. It prepares later cleaning, harmonization and source-selection discussions, but it does not rank datasets or recommend a V1 source.
<br>
<strong>EDA implication.</strong> No dataset recommendation should be inferred automatically from this matrix. The next decision should be based on the analytical question the project chooses to pursue.
</div>

<a id="conclusion-analytical-strategy"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Closing section</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Conclusion and analytical strategy</h2>
</div>

This foundation EDA confirms that ConflictLens has a strong and diverse data base for studying conflict-related violence. However, the datasets cannot be combined safely without first defining a clear analytical structure. They do not all describe the same type of object: some datasets describe countries over time, while others describe individual conflict events, population movements, political indicators, actors, or regional aggregates.

The main conclusion is therefore methodological. ConflictLens should not move directly from data inventory to modelling. The next stage should first build a stable analytical backbone, then use more detailed datasets as complementary modules.

### Strategic decision: two complementary analytical tracks

The project will be structured around two analytical tracks.

| Analytical track | Unit of analysis | Purpose | Project sequence |
|---|---|---|---|
| **Country-year analysis** | One country observed during one year | Compare countries over time and connect conflict intensity with political, socioeconomic, institutional and humanitarian indicators | **First track to build** |
| **Event-level analysis** | One conflict event or incident | Study local patterns, spatial concentration, escalation dynamics and forms of violence at a finer level | **Second track, after the country-year foundation is stable** |

This sequencing does not mean that event-level analysis is abandoned. On the contrary, it preserves the richness of event-level datasets while avoiding an unstable first integration layer. The country-year panel will provide the comparative frame. The event-level analysis will later provide the granular and spatial frame.

### Why start with a country-year panel?

A country-year panel means that each row represents a country in a specific year. For example, one row could describe Colombia in 2002, another row Syria in 2014, and another row Ukraine in 2022. This structure is easy to explain, easier to audit, and well suited to comparing conflict dynamics across countries and periods.

The EDA shows that this is the most reliable starting point because several major sources can be connected at this level. Conflict indicators, political indicators, socioeconomic indicators and displacement indicators can all be aligned around country and year, provided that country identifiers and time windows are harmonized carefully.

The recommended next notebook should therefore focus on building a clean country-year panel with a shared country identifier, a shared year variable, documented source definitions and transparent missing-value rules.

### Analytical axes for the country-year track

The first country-year analysis should cover two complementary axes.

#### Axis A — Conflict vulnerability and macro-context

This axis studies how conflict intensity evolves alongside broader country-level conditions. It is designed to answer questions such as:

> How do conflict intensity, political context, socioeconomic fragility and humanitarian displacement evolve together across countries over time?

This axis can connect conflict measures with indicators related to political institutions, economic development, demographic structure, social conditions and displacement. The goal is not to claim direct causality at this stage, but to identify meaningful patterns, contrasts and trajectories.

#### Axis B — Civilian exposure and violence against civilians

This axis focuses more directly on the human impact of conflict. It is designed to answer questions such as:

> Which countries and periods show the strongest concentration of civilian exposure, one-sided violence, civilian targeting and forced displacement?

This axis will help distinguish general conflict intensity from forms of violence that directly affect civilian populations. It can later be enriched with specialized datasets such as conflict-related sexual violence, displacement flows and event-level records of civilian-targeted violence.

### Recommended next steps

The next stage of the project should follow a clear sequence:

1. Build a harmonized country-year backbone using `country_iso3` and `year` as the reference keys.
2. Define the main temporal window for the country-year analysis, while documenting any source-specific limitations.
3. Assign a clear analytical role to each source: conflict outcomes, political context, socioeconomic context, humanitarian consequences, or specialized violence modules.
4. Produce descriptive country-year analyses for the two retained axes: macro conflict vulnerability and civilian exposure.
5. Only after this foundation is stable, move to the event-level track for spatial, local and incident-level analysis.

### Final takeaway

The foundation EDA confirms that ConflictLens should be built as a layered analytical project. The first layer will be a country-year panel designed for comparison, interpretation and communication. The second layer will preserve the detail of event-level data for later spatial and granular analysis.

This strategy keeps the project rigorous while remaining accessible: it starts with a structure that non-technical readers can understand, and it leaves room for deeper event-level analysis once the main analytical foundation is reliable.